# Демо 3 · Свій SLAM на KITTI і порівняння з бібліотеками
**Заняття 3 · до слайдів «Архітектура типової системи SLAM», «Побудова та оновлення факторного графа», «Loop closure (замикання петлі)», «Інтеграція обмежень замикання циклу», «ATE проти RPE», «Рекомендовані SLAM фреймворки під Deftech задачі»**

Будуємо стерео-SLAM сходинками й міряємо внесок кожної за **одним протоколом** — тим самим, яким оцінюють бібліотеки:

| Сходинка | Що додаємо | Що це лікує |
|---|---|---|
| 1 · VO кадр-до-кадру | KLT + стерео + PnP (demo2) | — базовий рівень |
| 2 · Ключові кадри + локальна карта | трекінг до 3D-точок кількох ключових кадрів (ORB) | коротший ланцюжок поз → повільніший дрейф |
| 3 · Розпізнавання місць | мішок візуальних слів (Хеммінг) + перевірка PnP | знаходимо, де ми вже були |
| 4 · Граф поз 4-DoF | Гаусс–Ньютон, ядро Коші | розподіляємо накопичену похибку по траєкторії |

Еталони: **KISS-ICP** (лідарна одометрія, ставиться через pip на Windows/macOS/Linux і запускається тут же), **ORB-SLAM3 stereo** (готові траєкторії з `tools/fetch_kitti.py`) і опубліковані результати з статей.

Мета — не «обіграти ORB-SLAM» (C++ із локальною BA і роками налагодження), а побачити, **що саме дає кожен модуль**, — бібліотека цього не покаже.

**Дані:** KITTI 00 (`python tools/fetch_kitti.py --no-lidar`), для словника — кадри 05, для KISS-ICP — повні скани лідара (див. `artifacts/kitti/README.md`). Пакети (`scipy`, `kiss-icp`) — у кореневому `requirements.txt`. **Час:** повна послідовність 00 — ≈15–20 хв на першому запуску, далі все з кешу в `outputs/`. Для швидкого показу поставте `SEQUENCE = '07'` (1101 кадр, одна петля в кінці).

In [ ]:
from pathlib import Path
import time, pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from tqdm.auto import tqdm

KITTI_ROOT = Path('../../artifacts/kitti')     # сюди кладе дані tools/fetch_kitti.py
SEQUENCE = '00'           # '07' — швидкий варіант для лекції
MAX_FRAMES = None         # напр. 2000 — швидший прогін; перша петля в 00 — на кадрі ≈ 1560, з меншим лімітом SLAM = VO
VOCAB_SEQUENCE = '05'     # словник для розпізнавання місць вчимо на ІНШІЙ послідовності — щоб не підглядати
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
REF_DIR = KITTI_ROOT / 'reference'                   # траєкторії ORB-SLAM3: tools/fetch_kitti.py

STYLE = {
    'GT':                  dict(color='#3b3b3b', ls='--', lw=1.8),
    'VO кадр-до-кадру':    dict(color='#eb6834', ls='-', lw=1.6),
    'VO + ключові кадри':  dict(color='#2a78d6', ls='-', lw=1.6),
    'SLAM (+ петлі)':      dict(color='#1baf7a', ls='-', lw=2.4),
    'KISS-ICP (лідар)':    dict(color='#4a3aa7', ls='-', lw=1.6),
    'ORB-SLAM3 stereo':    dict(color='#e87ba4', ls='-', lw=1.6),
}

## 1 · Дані

In [ ]:
def load_seq(root, seq):
    sd = Path(root) / 'sequences' / seq
    cal = {}
    for line in (sd / 'calib.txt').read_text().splitlines():
        if ':' in line:
            k, v = line.split(':', 1); cal[k.strip()] = np.array(v.split(), float).reshape(3, 4)
    K = cal['P0'][:, :3]; B = -cal['P1'][0, 3] / cal['P1'][0, 0]
    pp = Path(root) / 'poses' / f'{seq}.txt'
    gt = None
    if pp.exists():
        g = np.loadtxt(pp).reshape(-1, 3, 4); gt = np.tile(np.eye(4), (len(g), 1, 1)); gt[:, :3, :] = g
    return dict(root=Path(root), id=seq, dir=sd, K=K, B=B, gt=gt, n=len(list((sd / 'image_0').glob('*.png'))))


def load_pair(seq, i):
    l = cv2.imread(str(seq['dir'] / 'image_0' / f'{i:06d}.png'), cv2.IMREAD_GRAYSCALE)
    r = cv2.imread(str(seq['dir'] / 'image_1' / f'{i:06d}.png'), cv2.IMREAD_GRAYSCALE)
    return l, r


assert (KITTI_ROOT / 'sequences' / SEQUENCE / 'image_1').exists(), 'KITTI не знайдено: python tools/fetch_kitti.py --no-lidar'
SEQ = load_seq(KITTI_ROOT, SEQUENCE)
assert SEQ['gt'] is not None, 'для оцінки потрібна послідовність 00–10 (з GT)'
SEQ['n_full'] = SEQ['n']
if MAX_FRAMES:
    SEQ['n'] = min(SEQ['n'], MAX_FRAMES); SEQ['gt'] = SEQ['gt'][:SEQ['n']]
GT = SEQ['gt']
TAG = SEQUENCE if SEQ['n'] == SEQ['n_full'] else f"{SEQUENCE}_{SEQ['n']}"   # ім'я кешу: обрізаний прогін не плутаємо з повним
print(f"{SEQ['dir']}: {SEQ['n']} з {SEQ['n_full']} кадрів · f = {SEQ['K'][0, 0]:.1f} px · B = {SEQ['B']:.3f} м")

## 2 · Протокол оцінки: дві різні метрики (слайд «ATE проти RPE»)
- **ATE (APE)** — RMSE позицій після вирівнювання з GT. Міряє **глобальну узгодженість**: саме тут видно ефект замикання петель. Вирівнювання: SE(3) для стерео й лідара (`evo_ape --align`), Sim(3) для монокуляра (`--align --correct_scale`). Додатково показуємо ATE без вирівнювання — від спільного старту.
- **t_err / r_err** — офіційна метрика KITTI: відносна похибка на всіх відрізках 100…800 м, у % і °/100 м. Міряє **локальний дрейф** — тут VO і SLAM можна порівнювати чесно.

Порівнювати VO зі SLAM лише за ATE — помилка: петлі зменшують t_err помірно (у нас ≈20 %), а ATE — у рази. Числа з різних статей без знання протоколу оцінки (вирівнювання, які кадри враховано) теж не порівнюють.

In [ ]:
LENGTHS = (100, 200, 300, 400, 500, 600, 700, 800)


def path_dist(P):
    return np.r_[0, np.cumsum(np.linalg.norm(np.diff(P[:, :3, 3], axis=0), axis=1))]


def rot_angle(R):
    return np.arccos(np.clip((np.trace(R, axis1=-2, axis2=-1) - 1) / 2, -1, 1))


def kitti_errors(est, gt, step=10):
    # порт офіційного devkit: усі відрізки 100…800 м, старт кожні 10 кадрів
    d = path_dist(gt)
    inv_gt, inv_est = np.linalg.inv(gt), np.linalg.inv(est)
    t_all, r_all, per_len = [], [], {L: [] for L in LENGTHS}
    for i in range(0, len(gt), step):
        for L in LENGTHS:
            j = np.searchsorted(d, d[i] + L)
            if j >= len(gt):
                continue
            E = np.linalg.inv(inv_gt[i] @ gt[j]) @ (inv_est[i] @ est[j])
            t_all.append(np.linalg.norm(E[:3, 3]) / L); r_all.append(rot_angle(E[:3, :3]) / L); per_len[L].append(t_all[-1])
    return 100 * np.mean(t_all), np.degrees(np.mean(r_all)) * 100, {L: 100 * np.mean(v) for L, v in per_len.items() if v}


def umeyama(src, dst, with_scale=False):
    mu_s, mu_d = src.mean(0), dst.mean(0)
    S, D = src - mu_s, dst - mu_d
    U, sig, Vt = np.linalg.svd(D.T @ S / len(src))
    F = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        F[2, 2] = -1
    R = U @ F @ Vt
    s = np.trace(np.diag(sig) @ F) / S.var(0).sum() if with_scale else 1.0
    return s, R, mu_d - s * R @ mu_s


def ate(est, gt, align='se3'):
    e, g = est[:, :3, 3], gt[:, :3, 3]
    if align != 'none':
        s, R, t = umeyama(e, g, with_scale=(align == 'sim3'))
        e = s * e @ R.T + t
    err = np.linalg.norm(e - g, axis=1)
    return float(np.sqrt(np.mean(err ** 2))), err


def evaluate(est, gt=GT):
    t_err, r_err, per_len = kitti_errors(est, gt)
    a, err = ate(est, gt, 'se3')
    return dict(ate=a, ate_start=ate(est, gt, 'none')[0], t_err=t_err, r_err=r_err, per_len=per_len, err=err)


DIST = path_dist(GT)
print(f'довжина {DIST[-1] / 1000:.2f} км')

## 3 · Сходинка 1 — VO кадр-до-кадру
Той самий 3D-2D, що в demo2: KLT-треки, розріджене стерео для глибини, PnP + RANSAC. Поза — добуток відносних рухів **кожної** пари кадрів, тож похибка кожного кроку накопичується.

In [ ]:
LK = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))


def to_T(R, t):
    T = np.eye(4); T[:3, :3] = R; T[:3, 3] = np.ravel(t)
    return T


def track(prev, cur, n_features=800):
    p0 = cv2.goodFeaturesToTrack(prev, maxCorners=n_features, qualityLevel=0.01, minDistance=7, blockSize=7)
    if p0 is None:
        return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)
    p1, st1, _ = cv2.calcOpticalFlowPyrLK(prev, cur, p0, None, **LK)
    pb, st2, _ = cv2.calcOpticalFlowPyrLK(cur, prev, p1, None, **LK)
    ok = (st1.ravel() == 1) & (st2.ravel() == 1) & (np.linalg.norm(p0 - pb, axis=2).ravel() < 1.0)
    return p0.reshape(-1, 2)[ok], p1.reshape(-1, 2)[ok]


def stereo_depth(left, right, pts, K, B, max_depth_B):
    # глибина точок із розрідженого стерео (KLT ліво→право вздовж рядка); NaN — немає надійної відповідності
    Z = np.full(len(pts), np.nan)
    if len(pts) == 0:
        return Z
    p = pts.reshape(-1, 1, 2).astype(np.float32)
    pr, st1, _ = cv2.calcOpticalFlowPyrLK(left, right, p, p.copy(), flags=cv2.OPTFLOW_USE_INITIAL_FLOW, **LK)
    pb, st2, _ = cv2.calcOpticalFlowPyrLK(right, left, pr, None, **LK)
    d = (p - pr).reshape(-1, 2)
    ok = (st1.ravel() == 1) & (st2.ravel() == 1) & (np.linalg.norm(p - pb, axis=2).ravel() < 1.0) & (np.abs(d[:, 1]) < 1.0) & (d[:, 0] > 0.5)
    Z[ok] = K[0, 0] * B / d[ok, 0]
    Z[Z > max_depth_B * B] = np.nan
    return Z


def backproject(pts, Z, K):
    return np.stack([(pts[:, 0] - K[0, 2]) / K[0, 0] * Z, (pts[:, 1] - K[1, 2]) / K[1, 1] * Z, Z], 1)


def pnp(obj, img, K, min_inl=15, px=2.0):
    # PnP + RANSAC (EPnP) + уточнення LM; повертає T_cw: X_c = R·X_w + t
    if len(obj) < min_inl:
        return None, 0
    good, rv, tv, inl = cv2.solvePnPRansac(obj, img, K, None, iterationsCount=200, reprojectionError=px,
                                           confidence=0.999, flags=cv2.SOLVEPNP_EPNP)
    if not good or inl is None or len(inl) < min_inl:
        return None, 0
    i = inl.ravel()
    rv, tv = cv2.solvePnPRefineLM(obj[i], img[i], K, None, rv, tv)
    return to_T(cv2.Rodrigues(rv)[0], tv), len(i)


def run_f2f(seq, max_depth_B=40.0):
    K, B = seq['K'], seq['B']
    T, last, t0 = seq['gt'][0].copy(), np.eye(4), time.time()
    poses = [T.copy()]
    l0, r0 = load_pair(seq, 0)
    for i in tqdm(range(1, seq['n']), desc='VO кадр-до-кадру', unit='кадр'):
        l, r = load_pair(seq, i)
        p0, p1 = track(l0, l)
        ok = np.isfinite(Z0 := stereo_depth(l0, r0, p0, K, B, max_depth_B))
        Tr, _ = pnp(backproject(p0[ok], Z0[ok], K), p1[ok].astype(np.float64), K) if ok.sum() else (None, 0)
        Tr = last if Tr is None else Tr
        last = Tr; T = T @ np.linalg.inv(Tr); poses.append(T.copy())
        l0, r0 = l, r
    return dict(poses=np.array(poses), sec=time.time() - t0)


def cached(name, fn):
    path = OUT_DIR / f'slam_{TAG}_{name}.pkl'
    if path.exists():
        return pickle.loads(path.read_bytes())
    t0 = time.time(); res = fn(); path.write_bytes(pickle.dumps(res))
    print(f'{name}: {time.time() - t0:.0f} с, збережено {path}')
    return res


F2F = cached('f2f', lambda: run_f2f(SEQ))

## 4 · Сходинка 2 — ключові кадри й локальна карта
Замість «попередній кадр → поточний» поточний кадр трекаємо до **3D-точок кількох останніх ключових кадрів** (тут 8): ORB-дескриптори, пошук лише біля прогнозованої проєкції (модель сталої швидкості), PnP. Між ключовими кадрами поза не ланцюжиться — кожен кадр прив'язаний до тієї самої карти, тож похибка накопичується лише при створенні нового ключового кадру.

Новий ключовий кадр — коли інлаєрів < 150 або від останнього ключового > 6 м. Глибину беремо до 100·B: далекі точки слабкі для трансляції, але тримають обертання (див. demo2, розділ 2).

Ключовий кадр зберігає все потрібне для наступних сходинок: 3D-точки (у системі камери й світу), їхні дескриптори і **всі** ORB-ознаки кадру — для розпізнавання місць.

In [ ]:
ORB = cv2.ORB_create(nfeatures=2000, scaleFactor=1.2, nlevels=8, fastThreshold=12)
BF = cv2.BFMatcher(cv2.NORM_HAMMING)


def make_kf(idx, left, right, T_wc, K, B, max_depth_B):
    kp, des = ORB.detectAndCompute(left, None)
    pts = np.float32([k.pt for k in kp])
    Z = stereo_depth(left, right, pts, K, B, max_depth_B)
    ok = np.isfinite(Z)
    Xc = backproject(pts[ok], Z[ok], K)
    return dict(idx=idx, T=T_wc.copy(), Xc=Xc, Xw=Xc @ T_wc[:3, :3].T + T_wc[:3, 3], des=des[ok], pts_all=pts, des_all=des)


def match_ratio(dq, dt, max_dist=64, ratio=0.85):
    # найближчий сусід за Хеммінгом + тест Лоу
    out = []
    for m in BF.knnMatch(dq, dt, k=2):
        if len(m) == 2 and m[0].distance < max_dist and m[0].distance < ratio * m[1].distance:
            out.append((m[0].queryIdx, m[0].trainIdx))
    return out

In [ ]:
def track_local_map(pts, des, kfs, T_pred, K, radius=40.0):
    Xw = np.vstack([k['Xw'] for k in kfs]); D = np.vstack([k['des'] for k in kfs])
    T_cw = np.linalg.inv(T_pred)
    Xc = Xw @ T_cw[:3, :3].T + T_cw[:3, 3]
    uv = (Xc[:, :2] / np.maximum(Xc[:, 2:], 1e-6)) * [K[0, 0], K[1, 1]] + [K[0, 2], K[1, 2]]
    sel = [(q, t) for q, t in match_ratio(D, des) if Xc[q, 2] > 0.5 and np.linalg.norm(pts[t] - uv[q]) < radius]
    if not sel:
        return None, 0
    q, t = np.array(sel).T
    T_cw, n = pnp(Xw[q].astype(np.float64), pts[t].astype(np.float64), K)
    return (None, 0) if T_cw is None else (np.linalg.inv(T_cw), n)

In [ ]:
def run_kf_vo(seq, n_local=8, min_inl=150, max_dist=6.0, max_depth_B=100.0):
    K, B = seq['K'], seq['B']
    T, vel = seq['gt'][0].copy(), np.eye(4)
    l, r = load_pair(seq, 0)
    kfs = [make_kf(0, l, r, T, K, B, max_depth_B)]
    poses, ref_kf, inls, t0 = [T.copy()], [0], [0], time.time()
    bar = tqdm(range(1, seq['n']), desc='VO + ключові кадри', unit='кадр')
    for i in bar:
        l, r = load_pair(seq, i)
        kp, des = ORB.detectAndCompute(l, None)
        pts = np.float32([k.pt for k in kp])
        T_pred = T @ vel
        T_new, ninl = track_local_map(pts, des, kfs[-n_local:], T_pred, K)
        if T_new is None:
            T_new = T_pred                                  # втрата трекінгу: модель сталої швидкості
        vel = np.linalg.inv(T) @ T_new; T = T_new
        if ninl < min_inl or np.linalg.norm(T[:3, 3] - kfs[-1]['T'][:3, 3]) > max_dist:
            kfs.append(make_kf(i, l, r, T, K, B, max_depth_B))
        poses.append(T.copy()); ref_kf.append(len(kfs) - 1); inls.append(ninl)
        bar.set_postfix(ключових=len(kfs), inliers=ninl, refresh=False)
    return dict(poses=np.array(poses), ref_kf=np.array(ref_kf), inliers=np.array(inls), kfs=kfs, sec=time.time() - t0)


KFVO = cached('kfvo', lambda: run_kf_vo(SEQ))
KFS = KFVO['kfs']
print(f"ключових кадрів: {len(KFS)} з {SEQ['n']} · втрат трекінгу: {(KFVO['inliers'][1:] == 0).sum()}")
for name, P in [('VO кадр-до-кадру', F2F['poses']), ('VO + ключові кадри', KFVO['poses'])]:
    e = evaluate(P)
    print(f"{name:<20} t_err {e['t_err']:.2f} % · r_err {e['r_err']:.2f} °/100 м · ATE {e['ate']:.1f} м (SE3) / {e['ate_start']:.1f} м (від старту)")

## 5 · Сходинка 3 — розпізнавання місць
**Мішок візуальних слів.** Дескриптори ORB — 256 біт, і порівнюються **відстанню Хеммінга**. Тому словник будуємо k-majority-кластеризацією: центр кластера — побітова більшість. (OpenCV `BOWKMeansTrainer` рахує k-means з L2 по байтах — для бінарних дескрипторів це математично неправильно, а готових DBoW-обгорток під Python 3.11 немає.) Відстань Хеммінга між бітовими векторами `a, c ∈ {0,1}²⁵⁶` — одне матричне множення: `|a⊕c| = Σa + Σc − 2·a·c`.

Кадр → гістограма слів, зважена IDF (рідкісні слова важать більше), L1-нормована. Схожість як у DBoW: `s = 1 − ½‖v₁ − v₂‖₁`. Словник учимо на **іншій** послідовності (`VOCAB_SEQUENCE`), інакше він «знає» місця заздалегідь.

In [ ]:
def bits(D):
    return np.unpackbits(D, axis=1).astype(np.float32)


def assign_words(D, Cb, chunk=20000):
    Db = bits(D)
    return np.concatenate([(Db[i:i + chunk].sum(1, keepdims=True) + Cb.sum(1)[None] - 2 * Db[i:i + chunk] @ Cb.T).argmin(1)
                           for i in range(0, len(Db), chunk)])


def train_vocab(D, k=1000, iters=10, seed=0):
    rng = np.random.default_rng(seed)
    Db = bits(D)
    Cb = Db[rng.choice(len(Db), k, replace=False)].copy()
    for _ in tqdm(range(iters), desc=f'словник: k-majority, k = {k}', unit='ітер'):
        a = assign_words(D, Cb)
        for w in range(k):
            m = a == w
            Cb[w] = (Db[m].mean(0) > 0.5) if m.any() else Db[rng.integers(len(Db))]   # побітова більшість
    return Cb


def build_vocab(seq_id, step=10, k=1000, sample=120000):
    seq = load_seq(KITTI_ROOT, seq_id)
    descs = [ORB.detectAndCompute(load_pair(seq, i)[0], None)[1]
             for i in tqdm(range(0, seq['n'], step), desc=f'словник: ORB на {seq_id}', unit='кадр')]
    D = np.vstack(descs)
    D = D[np.random.default_rng(0).choice(len(D), min(sample, len(D)), replace=False)]
    Cb = train_vocab(D, k)
    df = np.zeros(k)
    for d in descs:
        df[np.unique(assign_words(d, Cb))] += 1
    return dict(Cb=Cb, idf=np.log(len(descs) / np.maximum(df, 1)))

In [ ]:
def bow_vector(des, vocab):
    v = np.bincount(assign_words(des, vocab['Cb']), minlength=len(vocab['Cb'])).astype(np.float64) * vocab['idf']
    return v / max(v.sum(), 1e-12)


def bow_score(v1, v2):
    return 1 - 0.5 * np.abs(v1 - v2).sum()

In [ ]:
vp = OUT_DIR / f'bow_{VOCAB_SEQUENCE}.npz'
if vp.exists():
    z = np.load(vp); VOCAB = dict(Cb=z['Cb'].astype(np.float32), idf=z['idf'])
else:
    t0 = time.time(); VOCAB = build_vocab(VOCAB_SEQUENCE)
    np.savez(vp, Cb=VOCAB['Cb'].astype(np.uint8), idf=VOCAB['idf']); print(f'словник: {time.time() - t0:.0f} с')
V = np.array([bow_vector(k['des_all'], VOCAB) for k in tqdm(KFS, desc='BoW ключових кадрів', unit='KF')])
S = 1 - 0.5 * np.abs(V[:, None, :] - V[None, :, :]).sum(-1)            # схожість усіх пар ключових кадрів
KF_IDX = np.array([k['idx'] for k in KFS])
GT_KF = GT[KF_IDX]
D_GT = np.linalg.norm(GT_KF[:, None, :3, 3] - GT_KF[None, :, :3, 3], axis=-1)

**Кандидат і перевірка.** Для кожного ключового кадру беремо найсхожіший старий (давніше 300 кадрів). Оцінку нормуємо на схожість із сусіднім ключовим кадром (`η = s / s_prev`, як у DBoW) — так поріг не залежить від «текстурності» місця. Кандидата з `η ≥ 0,6` перевіряємо **геометрично**: 3D-точки старого кадру → пікселі поточного, PnP + RANSAC; петля приймається лише з ≥ 60 інлаєрами. Перевірка дає і відносну позу — вимірювання для графа.

In [ ]:
def detect_loops(kfs, V, K, min_gap=300, min_eta=0.6, min_inl=60):
    loops = []
    bar = tqdm(range(1, len(kfs)), desc='пошук петель', unit='KF')
    for j in bar:
        old = [i for i in range(len(kfs)) if kfs[i]['idx'] < kfs[j]['idx'] - min_gap]
        if not old:
            continue
        s = np.array([bow_score(V[i], V[j]) for i in old])
        b = int(np.argmax(s)); i = old[b]
        eta = s[b] / max(bow_score(V[j], V[j - 1]), 1e-6)
        if eta < min_eta:
            continue
        m = match_ratio(kfs[i]['des'], kfs[j]['des_all'])
        if len(m) < min_inl:
            continue
        q, t = np.array(m).T
        T_ji, n = pnp(kfs[i]['Xc'][q].astype(np.float64), kfs[j]['pts_all'][t].astype(np.float64), K, min_inl=min_inl)
        if T_ji is not None:
            loops.append(dict(i=i, j=j, T_ji=T_ji, inliers=n, eta=eta))
            bar.set_postfix(петель=len(loops), refresh=False)
    return loops


t0 = time.time(); LOOPS = detect_loops(KFS, V, SEQ['K']); T_LOOPS = time.time() - t0
for lp in LOOPS:                                    # перевірка за GT: чи правильна виміряна відносна поза
    E = np.linalg.inv(np.linalg.inv(GT_KF[lp['j']]) @ GT_KF[lp['i']]) @ lp['T_ji']
    lp['err_t'], lp['err_r'] = np.linalg.norm(E[:3, 3]), np.degrees(rot_angle(E[:3, :3]))
    lp['true'] = lp['err_t'] < 1.5 and lp['err_r'] < 2.0
revisit = np.array([np.any((D_GT[j, :j] < 5) & (KF_IDX[:j] < KF_IDX[j] - 300)) for j in range(len(KFS))])
found = np.zeros(len(KFS), bool); found[[lp['j'] for lp in LOOPS if lp['true']]] = True
print(f"петель знайдено: {len(LOOPS)}, з них правильних {sum(lp['true'] for lp in LOOPS)} · "
      f"ключових кадрів із повторним проїздом за GT: {revisit.sum()}, знайдено серед них: {(found & revisit).sum()} "
      f"({100 * (found & revisit).sum() / max(revisit.sum(), 1):.0f} %)")
if LOOPS:
    et = np.array([lp['err_t'] for lp in LOOPS]); er = np.array([lp['err_r'] for lp in LOOPS])
    print(f'розбіжність виміряної відносної пози з GT: трансляція медіана {np.median(et):.2f} м (макс {et.max():.2f}), '
          f'обертання медіана {np.median(er):.2f}° (макс {er.max():.2f}°)')

In [ ]:
far = np.abs(KF_IDX[:, None] - KF_IDX[None]) > 300                     # пари, далекі в часі
fig, ax = plt.subplots(1, 2, figsize=(15, 6.8), layout='constrained')
im = ax[0].imshow(np.where(far, S, np.nan), cmap='Blues', vmin=np.nanpercentile(S[far], 50), vmax=np.nanpercentile(S[far], 99.9))
ax[0].set_title('Схожість BoW між ключовими кадрами\n(показано лише пари, далекі в часі: > 300 кадрів)')
ax[0].set_xlabel('ключовий кадр'); ax[0].set_ylabel('ключовий кадр')
fig.colorbar(im, ax=ax[0], shrink=0.8, label='схожість (шкала від медіани)')
ax[1].imshow(np.where(far, D_GT < 5, np.nan), cmap='Greys', vmin=0, vmax=1.4)
ax[1].set_title('GT: справжні повторні проїзди (< 5 м) і знайдені петлі'); ax[1].set_xlabel('ключовий кадр')
for a in ax:
    a.plot([], [], 'o', mfc='none', mec='#eb6834', ms=6, label='знайдена петля')
    for lp in LOOPS:
        a.plot(lp['i'], lp['j'], 'o', mfc='none', mec='#eb6834' if lp['true'] else '#e34948', ms=6, mew=1.4)
    a.legend(loc='lower left', fontsize=9)
plt.show()

Яскраві смуги ліворуч, паралельні до діагоналі, мають збігатися зі смугами праворуч — це повторні проїзди в тому ж напрямку, і саме там стоять знайдені петлі. Смуги, перпендикулярні до діагоналі, означають проїзд того самого місця **назустріч**: камера бачить інший бік вулиці, і візуальна схожість низька (див. послідовність 08).

**Обережно з «правдою».** GT KITTI — це GPS/INS, і на повторних проїздах він сам має похибку порядку метра. На 00 частина петель розходиться з GT на ≈1 м при збігу обертання до десятих градуса і сотнях інлаєрів — схоже, що тут неточний радше GT, ніж PnP. Тому поріг «правильної» петлі — 1,5 м і 2°, і порівнювати ATE систем нижче ≈1 м на KITTI вже мало сенсу.

## 6 · Сходинка 4 — граф поз 4-DoF
Вузли — ключові кадри, ребра — відносні пози: послідовні (з VO) і петлі (з перевірки PnP). Оптимізуємо лише **x, y, z і курс ψ**, а крен і тангаж беремо з VO й не змінюємо. У VIO (VINS-Mono/Fusion) це точно: гравітація з IMU робить крен і тангаж спостережуваними, і дрейфують лише 4 ступені свободи. У нашої стерео-VO гравітації немає, тож тут це **спрощення**: дрейф крену й тангажу VO (на 00 — до ≈8°, медіана ≈2°) переходить у результат без змін. Позиції петлі виправляють напряму, тому ATE усе одно падає; повне рішення — 6-DoF граф (завдання 4 наприкінці ноутбука).

Для ребра i→j з виміряним зсувом `t̂` (у системі i) і приростом курсу `ψ̂`:

`r_t = R_iᵀ (p_j − p_i) − t̂`,   `r_ψ = ψ_j − ψ_i − ψ̂`,   де `R_i = R_y(ψ_i) · R_rp,i`.

Гаусс–Ньютон із розрідженим `H = Σ Jᵀ W J`; на петлях — ядро Коші, щоб хибна петля не зламала карту. Перший вузол зафіксовано.

In [ ]:
def yaw_of(R):
    return np.arctan2(R[0, 2], R[2, 2])


def Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def dRy(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[-s, 0, c], [0, 0, 0], [-c, 0, -s]])


def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def build_edges(kfs, loops, sig_odo=(0.1, np.radians(0.2)), sig_loop=(0.3, np.radians(1.0))):
    def rel(Ti, Tj):
        return Ti[:3, :3].T @ (Tj[:3, 3] - Ti[:3, 3]), wrap(yaw_of(Tj[:3, :3]) - yaw_of(Ti[:3, :3]))
    edges = []
    for k in range(len(kfs) - 1):
        t, dpsi = rel(kfs[k]['T'], kfs[k + 1]['T'])
        edges.append(dict(i=k, j=k + 1, t=t, psi=dpsi, st=sig_odo[0] + 0.01 * np.linalg.norm(t), spsi=sig_odo[1], loop=False))
    for lp in loops:
        Ti = kfs[lp['i']]['T']; M = np.linalg.inv(lp['T_ji'])      # поза j у системі i
        edges.append(dict(i=lp['i'], j=lp['j'], t=M[:3, 3], psi=wrap(yaw_of(Ti[:3, :3] @ M[:3, :3]) - yaw_of(Ti[:3, :3])),
                          st=sig_loop[0], spsi=sig_loop[1], loop=True))
    return edges

In [ ]:
def edge_residual_jac(p_i, p_j, psi_i, psi_j, Rrp_i, e):
    RiT = Rrp_i.T @ Ry(psi_i).T
    dp = p_j - p_i
    r = np.r_[RiT @ dp - e['t'], wrap(psi_j - psi_i - e['psi'])]
    Ji, Jj = np.zeros((4, 4)), np.zeros((4, 4))
    Ji[:3, :3] = -RiT; Ji[:3, 3] = Rrp_i.T @ dRy(psi_i).T @ dp; Ji[3, 3] = -1
    Jj[:3, :3] = RiT; Jj[3, 3] = 1
    return r, Ji, Jj

In [ ]:
def optimize_4dof(kfs, edges, iters=20, kernel_c=3.0):
    n = len(kfs)
    p = np.array([k['T'][:3, 3] for k in kfs])
    psi = np.array([yaw_of(k['T'][:3, :3]) for k in kfs])
    Rrp = np.array([Ry(-a) @ k['T'][:3, :3] for a, k in zip(psi, kfs)])     # крен/тангаж — з VO, не оптимізуємо
    blk = np.add.outer(np.arange(4), np.zeros(4, int))                      # індекси 4×4 блоку
    hist = []
    bar = tqdm(desc=f'оптимізація графа (≤ {iters} ітер.)', unit='ітер')   # без total: зупиняється, щойно збіглось
    for it in range(iters):
        rows, cols, vals, b, cost = [], [], [], np.zeros(4 * n), 0.0
        for e in edges:
            i, j = e['i'], e['j']
            r, Ji, Jj = edge_residual_jac(p[i], p[j], psi[i], psi[j], Rrp[i], e)
            W = np.diag([1 / e['st'] ** 2] * 3 + [1 / e['spsi'] ** 2])
            s2 = r @ W @ r
            w = 1.0 / (1.0 + s2 / kernel_c ** 2) if e['loop'] else 1.0      # ядро Коші лише на петлях
            cost += w * s2; W = w * W
            for a, Ja in ((i, Ji), (j, Jj)):
                b[4 * a:4 * a + 4] += Ja.T @ W @ r
                for c, Jc in ((i, Ji), (j, Jj)):
                    rows.append((4 * a + blk).ravel()); cols.append((4 * c + blk.T).ravel()); vals.append((Ja.T @ W @ Jc).ravel())
        rows.append(np.arange(4)); cols.append(np.arange(4)); vals.append(np.full(4, 1e12))   # фіксуємо перший вузол
        H = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(4 * n, 4 * n))
        dx = spla.spsolve(H, -b).reshape(n, 4)
        p += dx[:, :3]; psi = wrap(psi + dx[:, 3]); hist.append(cost)
        bar.update(); bar.set_postfix(cost=f'{cost:.3g}', max_dx=f'{np.abs(dx).max():.1e}')
        if np.abs(dx).max() < 1e-5:
            break
    bar.close()
    T = np.tile(np.eye(4), (n, 1, 1))
    for k in range(n):
        T[k, :3, :3] = Ry(psi[k]) @ Rrp[k]; T[k, :3, 3] = p[k]
    return T, hist


def correct_frames(poses, ref_kf, kfs, T_kf_new):
    # кожен кадр «їде» разом зі своїм опорним ключовим кадром
    return np.array([T_kf_new[k] @ np.linalg.inv(kfs[k]['T']) @ P for P, k in zip(poses, ref_kf)])


t0 = time.time()
T_KF, cost_hist = optimize_4dof(KFS, build_edges(KFS, LOOPS))
SLAM = correct_frames(KFVO['poses'], KFVO['ref_kf'], KFS, T_KF)
T_PGO = time.time() - t0
print(f'граф: {len(KFS)} вузлів, {len(KFS) - 1 + len(LOOPS)} ребер · {len(cost_hist)} ітерацій · {T_PGO:.1f} с')


def tilt_err(P, gt=GT):
    # кут між віссю y камери (≈ «вниз») в оцінці і в GT, у системі камери — не залежить від похибки курсу
    y_est, y_gt = P[:, 1, :3], gt[:, 1, :3]
    return np.degrees(np.arccos(np.clip(np.sum(y_est * y_gt, 1), -1, 1)))


for name, P in [('до графа', KFVO['poses']), ('після графа', SLAM)]:
    e = tilt_err(P)
    print(f'похибка крену/тангажу {name}: медіана {np.median(e):.2f}°, макс {e.max():.2f}° — 4-DoF граф її не виправляє')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 8), layout='constrained')
for a, (title, P, Tk) in zip(ax, [('VO + ключові кадри (до графа)', KFVO['poses'], np.array([k['T'] for k in KFS])),
                                  ('SLAM: після оптимізації графа', SLAM, T_KF)]):
    a.plot(GT[:, 0, 3], GT[:, 2, 3], **STYLE['GT'], label='GT')
    a.plot(P[:, 0, 3], P[:, 2, 3], **(STYLE['VO + ключові кадри'] if 'до' in title else STYLE['SLAM (+ петлі)']), label=title.split(':')[0])
    for lp in LOOPS:
        a.plot(Tk[[lp['i'], lp['j']], 0, 3], Tk[[lp['i'], lp['j']], 2, 3], color='#1baf7a' if lp['true'] else '#e34948', lw=1.2, alpha=0.8)
    a.set_aspect('equal', adjustable='datalim'); a.grid(alpha=0.25); a.set_xlabel('x, м'); a.set_ylabel('z, м')
    a.set_title(f"{title} · ATE {evaluate(P)['ate']:.1f} м"); a.legend(loc='best', fontsize=9)
ax[0].plot([], [], color='#1baf7a', label='петля (правильна)'); ax[0].plot([], [], color='#e34948', label='петля (хибна)'); ax[0].legend(fontsize=9)
plt.show()

## 7 · Еталони
**KISS-ICP** — лідарна одометрія (без петель), той самий набір KITTI. Викликаємо через Python API: консольний `kiss_icp_pipeline` на Windows падає на створенні символьного посилання `results/latest` (потрібні права адміністратора або режим розробника). Пози з системи лідара переводимо в систему лівої камери через `Tr` з `calib.txt` — інакше порівняння з GT некоректне. Для цього потрібен **повний** velodyne (≈ 6 ГБ; `fetch_kitti.py` дає лише 500 сканів для demo1, а це 359 м маршруту без жодної петлі), тому студентам готова траєкторія приходить у `kitti_reference.zip` → `reference/kiss_icp_00.txt`.

**ORB-SLAM3 stereo** — класичний SLAM (C++, локальна BA, петлі). Збирати його не треба: викладач один раз проганяє `tools/orbslam3_kitti.sh` (Ubuntu 22.04 або WSL2), а готові траєкторії приходять архівом через `tools/fetch_kitti.py` в `artifacts/kitti/reference/`. ORB-SLAM багатопотоковий і недетермінований, тож беремо медіану з кількох прогонів. Якщо трекінг губився, ORB-SLAM3 пропускає кадри й починає нову карту (Atlas) зі своєю системою координат — такий прогін уже не є однією неперервною траєкторією, тому його відкидаємо.

In [ ]:
def run_kiss_icp(seq):
    from kiss_icp.kiss_icp import KissICP
    from kiss_icp.config import load_config
    from kiss_icp.datasets.kitti import KITTIOdometryDataset
    ds = KITTIOdometryDataset(str(seq['root']), seq['id'])
    odo, P, t0 = KissICP(load_config(None)), np.zeros((seq['n'], 4, 4)), time.time()
    for i in tqdm(range(seq['n']), desc='KISS-ICP', unit='скан'):
        odo.register_frame(*ds[i]); P[i] = odo.last_pose
    return dict(poses=seq['gt'][0] @ ds.apply_calibration(P), sec=time.time() - t0)   # лідар → камера cam0


def kiss_icp_ref(seq):
    return REF_DIR / f"kiss_icp_{seq['id']}.txt"


def save_kiss_icp(res, seq):
    # готова траєкторія для студентів: повний velodyne (~6 ГБ) їм тоді не потрібен; пакує tools/pack_kitti.py
    f = kiss_icp_ref(seq); f.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(f, res['poses'][:, :3, :].reshape(len(res['poses']), 12), fmt='%.9e',
               header=f"KISS-ICP {seq['id']}: {len(res['poses'])} frames, cam0 frame (KITTI poses format), sec={res['sec']:.1f}")


def load_kiss_icp(seq):
    f = kiss_icp_ref(seq)
    if not f.exists():
        return None
    raw = np.loadtxt(f).reshape(-1, 3, 4)
    if len(raw) != seq['n_full']:
        print(f'{f.name}: {len(raw)} поз замість {seq["n_full"]} — пропускаю'); return None
    raw = raw[:seq['n']]                                # MAX_FRAMES: перші n кадрів повної траєкторії
    P = np.tile(np.eye(4), (len(raw), 1, 1)); P[:, :3, :] = raw
    sec = float(f.read_text().split('sec=', 1)[1].split()[0])            # час повного прогону
    return dict(poses=P, sec=sec * seq['n'] / seq['n_full'])


def load_orbslam3(seq):
    # медіанний за t_err прогін; прогони з втратою трекінгу (менше рядків, ніж кадрів) відкидаємо
    runs = []
    for f in sorted(REF_DIR.glob(f"orbslam3_stereo_{seq['id']}_run*.txt")):
        raw = np.loadtxt(f).reshape(-1, 3, 4)
        if len(raw) != seq['n_full']:
            print(f'{f.name}: {len(raw)} поз замість {seq["n_full"]} — трекінг губився, нова карта; пропускаю'); continue
        raw = raw[:seq['n']]
        P = np.tile(np.eye(4), (len(raw), 1, 1)); P[:, :3, :] = raw
        runs.append(seq['gt'][0] @ P)
    if not runs:
        return None
    t = [kitti_errors(P, seq['gt'])[0] for P in runs]
    return dict(poses=runs[int(np.argsort(t)[len(t) // 2])], n_runs=len(runs), t_runs=t)


REFS = {}
n_lidar = len(list((SEQ['dir'] / 'velodyne').glob('*.bin')))
kiss = load_kiss_icp(SEQ)
if kiss:                                            # готова траєкторія з reference/ — повний velodyne не потрібен
    REFS['KISS-ICP (лідар)'] = kiss
    print(f'KISS-ICP: готова траєкторія {kiss_icp_ref(SEQ).name} (перерахувати: видалити її й мати повний velodyne)')
elif n_lidar >= SEQ['n']:                           # повний velodyne (офіційний data_odometry_velodyne) — рахуємо самі
    try:
        REFS['KISS-ICP (лідар)'] = cached('kiss_icp', lambda: run_kiss_icp(SEQ))
        if SEQ['n'] == SEQ['n_full']:
            save_kiss_icp(REFS['KISS-ICP (лідар)'], SEQ)
    except ImportError:
        print('kiss-icp не встановлено (pip install kiss-icp) — лідарний еталон пропущено')
else:
    print(f'KISS-ICP: сканів LiDAR {n_lidar} з {SEQ["n"]} і немає {kiss_icp_ref(SEQ)} — python tools/fetch_kitti.py (частина reference)')
orb = load_orbslam3(SEQ)
if orb:
    REFS['ORB-SLAM3 stereo'] = orb
    print(f"ORB-SLAM3: {orb['n_runs']} прогонів, t_err {min(orb['t_runs']):.2f}…{max(orb['t_runs']):.2f} % — беремо медіанний")
else:
    print(f'ORB-SLAM3: файлів у {REF_DIR} немає — python tools/fetch_kitti.py (частина reference)')

## 8 · Порівняння

In [ ]:
RESULTS = {'VO кадр-до-кадру': dict(poses=F2F['poses'], sec=F2F['sec'], sensor='стерео', loops='ні'),
           'VO + ключові кадри': dict(poses=KFVO['poses'], sec=KFVO['sec'], sensor='стерео', loops='ні'),
           'SLAM (+ петлі)': dict(poses=SLAM, sec=KFVO['sec'] + T_LOOPS + T_PGO, sensor='стерео', loops=f'{len(LOOPS)}')}
for name, r in REFS.items():
    RESULTS[name] = dict(poses=r['poses'], sec=r.get('sec', np.nan), sensor='лідар' if 'лідар' in name else 'стерео',
                         loops='ні' if 'KISS' in name else 'так')
EVAL = {k: evaluate(v['poses']) for k, v in RESULTS.items()}

print(f"KITTI {SEQUENCE}: {DIST[-1] / 1000:.2f} км, {SEQ['n']} кадрів\n")
print(f"{'система':<22}{'сенсор':>8}{'петлі':>7}{'t_err, %':>10}{'r_err, °/100м':>15}{'ATE SE3, м':>12}{'ATE старт, м':>14}{'мс/кадр':>9}")
print('─' * 97)
for k, e in EVAL.items():
    r = RESULTS[k]
    print(f"{k:<22}{r['sensor']:>8}{r['loops']:>7}{e['t_err']:>10.2f}{e['r_err']:>15.2f}{e['ate']:>12.2f}{e['ate_start']:>14.2f}{1000 * r['sec'] / SEQ['n']:>9.0f}")
if SEQUENCE == '00' and SEQ['n'] == SEQ['n_full']:   # опубліковані числа — для повної 00
    print('\nОпубліковано для KITTI 00 (не наш запуск; ATE ORB-SLAM2 — RMSE після жорсткого вирівнювання за Sturm et al. 2012, порівнюйте з колонкою «ATE SE3»):')
    for row in [('ORB-SLAM2 stereo, з петлями', '0.70', '0.25', '1.3'), ('ORB-SLAM2 stereo, петлі й global BA вимкнені (перезапуск у статті Stereo DSO)', '0.83', '0.29', '—'),
                ('Stereo DSO (VO)', '0.84', '0.26', '—'), ('VISO2-S (VO кадр-до-кадру)', '1.53', '0.64', '—'),
                ('SOFT2 (VO, №1 у таблиці лідерів)', '0.68', '0.28', '—'), ('KISS-ICP (лідар)', '0.51', '—', '—')]:
        print(f'  {row[0]:<76} t_err {row[1]:>5} %   r_err {row[2]:>5}   ATE {row[3]:>4} м')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(GT[:, 0, 3], GT[:, 2, 3], **STYLE['GT'], label='GT')
for k, r in RESULTS.items():
    ax.plot(r['poses'][:, 0, 3], r['poses'][:, 2, 3], **STYLE[k], label=f"{k} · ATE {EVAL[k]['ate']:.1f} м")
ax.set_aspect('equal', adjustable='datalim'); ax.grid(alpha=0.25); ax.legend(loc='best', fontsize=9)
ax.set_xlabel('x, м'); ax.set_ylabel('z, м'); ax.set_title(f'KITTI {SEQUENCE}: траєкторії (вид зверху)')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5), layout='constrained')
for k, e in EVAL.items():
    ax[0].plot(DIST / 1000, e['err'], **{**STYLE[k], 'lw': 1.6}, label=k)
    L = sorted(e['per_len']); ax[1].plot(L, [e['per_len'][l] for l in L], marker='o', ms=5, **{**STYLE[k], 'lw': 1.6}, label=k)
ax[0].set_xlabel('пройдено, км'); ax[0].set_ylabel('похибка позиції після SE(3), м'); ax[0].grid(alpha=0.25); ax[0].legend(fontsize=9)
ax[0].set_title('ATE вздовж шляху: петлі «стягують» похибку')
ax[1].set_xlabel('довжина відрізка, м'); ax[1].set_ylabel('t_err, %'); ax[1].grid(alpha=0.25); ax[1].set_ylim(0, None)
ax[1].set_title('Метрика KITTI: локальний дрейф за довжиною відрізка')
plt.show()

## Висновки
- **Ключові кадри й локальна карта** зменшують локальний дрейф (t_err) — менше ланок у ланцюжку поз.
- **Петлі + граф поз** зменшують t_err помірно (у нас ≈20 %: 1,49 → 1,14 % на 00), а ATE — у кілька разів (9,2 → 2,4 м): похибка розподіляється по всій траєкторії, і карта стає глобально узгодженою. Опубліковані числа ORB-SLAM2 узгоджуються з цим (0,83 % без петель — перезапуск у статті Stereo DSO; 0,70 % і ATE 1,3 м з петлями — стаття ORB-SLAM2), але це різні запуски, тож різницю t_err не варто трактувати як точний вимір.
- **4-DoF граф** виправляє позиції й курс, але не крен і тангаж: для стерео без IMU це спрощення, і його видно в числах розділу 6.
- **Розпізнавання місць** знаходить не всі повторні проїзди: проїзд назустріч і мала кількість спільних ознак його обманюють. Тому геометрична перевірка обов'язкова, а ядро Коші — страховка від того, що пройде.
- **Лідар** (KISS-ICP) без петель уже має t_err на рівні найкращих стерео-систем — інша модальність, інша верхня межа. Камера дешевша, пасивна й легша, тож усе це має ціну.
- **Час** Python проти C++ не порівнюємо: показово лише, яка сходинка скільки коштує в нашому коді.
- Чого в нас немає порівняно з ORB-SLAM: локальної BA (уточнює і пози, і точки), злиття дубльованих точок карти після петлі, повторної локалізації після втрати трекінгу.

## Для самостійного дослідження (необов'язково)
1. **Поріг η і хибні петлі:** опустіть `min_eta` до 0,3 і `min_inl` до 20. Скільки хибних петель пройде? Що з ATE з ядром Коші і без нього (`kernel_c = 1e9`)?
2. **Послідовність 08:** усі повторні проїзди там — назустріч. Запустіть і поясніть, чому петель не знаходиться. Що змінить лідарне розпізнавання місць (Scan Context) або кругова камера?
3. **Словник:** порівняйте словник, навчений на 05, зі словником, навченим на самій 00. Чому другий — нечесний?
4. Додатково: замініть 4-DoF на повний 6-DoF граф (SE(3), лінеаризація в алгебрі Лі) і порівняйте на 00, де перепад висоти 26 м.